# Prospect Measurements + Intangibles

This notebook combines three prospect-evaluation layers:

1. Draft projection and college/international stats from `draft_prospects_2026_seed.csv`.
2. Official NBA combine measurements where available from `nba_draft_combine_anthro_2026_27.csv`.
3. Optional manual measurement overrides from `draft_prospect_measurements_intangibles_2026_seed.csv`.

The output is a single `prospect_evaluation_table` that can be joined into the draft/free-agency scenario model.

## Combine Source Reference

Official combine anthropometric measurements should be referenced from NBA Stats' Draft Combine Anthro page:

https://www.nba.com/stats/draft/combine-anthro

The local file `nba_draft_combine_anthro_2026_27.csv` is a snapshot pulled through the `nba_api` `DraftCombineStats` endpoint with `season_all_time='2026-27'`, which corresponds to the current NBA Stats draft-combine anthropometric table. The NBA page includes the combine sections for Draft History, Anthro, Strength & Agility, Shooting Drills, Non-Stationary Shooting, and Spot Up Shooting.

## Why Measurements Matter

College production tells us what a prospect did. Measurements help estimate how that production might translate. For the Bulls specifically, wingspan, standing reach, weight, vertical pop, and agility matter because the roster needs more defensive size, rim pressure, and reliable two-way players.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "data").exists():
    project_dir = project_dir / "fresh_start"

data_dir = project_dir / "data"
prospect_path = data_dir / "draft_prospects_2026_seed.csv"
combine_path = data_dir / "nba_draft_combine_anthro_2026_27.csv"
manual_measurements_path = data_dir / "draft_prospect_measurements_intangibles_2026_seed.csv"
output_path = data_dir / "prospect_evaluation_table_2026.csv"
combine_source_url = "https://www.nba.com/stats/draft/combine-anthro"

prospects = pd.read_csv(prospect_path)
combine = pd.read_csv(combine_path)
manual = pd.read_csv(manual_measurements_path)

print(prospects.shape, combine.shape, manual.shape)

(18, 27) (78, 47) (18, 10)


In [2]:
def normalize_player_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(char for char in name if not unicodedata.combining(char))
    for char in [".", "'", "’", "‘", "`"]:
        name = name.replace(char, "")
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", name)
    return re.sub(r"\s+", " ", name).strip()

def z_score(series, lower_is_better=False):
    numeric = pd.to_numeric(series, errors="coerce")
    std = numeric.std(ddof=0)
    if pd.isna(std) or std == 0:
        z = pd.Series(0, index=series.index)
    else:
        z = (numeric - numeric.mean()) / std
    return -z if lower_is_better else z

def coalesce_columns(df, first, second):
    return df[first].combine_first(df[second])

## Merge Prospect Stats With Measurements

The official combine table will only match prospects who are present in the NBA endpoint snapshot. Manual rows fill gaps for the rest of the current prospect pool. Manual fields should be updated as you collect more reliable sources.

In [3]:
prospects = prospects.copy()
combine = combine.copy()
manual = manual.copy()

prospect_name_aliases = {
    "aj dybantsa": "anicet dybantsa",
    "nate ament": "nathaniel ament",
    "chris cenac": "christopher cenac",
    "mikel brown": "christopher brown",
}

prospects["NAME_KEY"] = prospects["PLAYER_NAME"].apply(normalize_player_name).replace(prospect_name_aliases)
combine["NAME_KEY"] = combine["PLAYER_NAME"].apply(normalize_player_name)
manual["NAME_KEY"] = manual["PLAYER_NAME"].apply(normalize_player_name)

combine_keep = [
    "NAME_KEY", "PLAYER_NAME", "POSITION", "HEIGHT_WO_SHOES", "WEIGHT", "WINGSPAN",
    "STANDING_REACH", "MAX_VERTICAL_LEAP", "LANE_AGILITY_TIME", "THREE_QUARTER_SPRINT",
    "HEIGHT_WO_SHOES_FT_IN", "WINGSPAN_FT_IN", "STANDING_REACH_FT_IN"
]

manual_keep = [
    "NAME_KEY", "HEIGHT_WO_SHOES", "WEIGHT", "WINGSPAN", "STANDING_REACH", "MAX_VERTICAL_LEAP",
    "LANE_AGILITY_TIME", "THREE_QUARTER_SPRINT", "MEASUREMENT_SOURCE", "MEASUREMENT_NOTES"
]

evaluation = prospects.merge(
    combine[combine_keep],
    on="NAME_KEY",
    how="left",
    suffixes=("", "_COMBINE"),
).merge(
    manual[manual_keep],
    on="NAME_KEY",
    how="left",
    suffixes=("_COMBINE", "_MANUAL"),
)

measurement_fields = [
    "HEIGHT_WO_SHOES", "WEIGHT", "WINGSPAN", "STANDING_REACH", "MAX_VERTICAL_LEAP",
    "LANE_AGILITY_TIME", "THREE_QUARTER_SPRINT"
]

for field in measurement_fields:
    evaluation[field] = coalesce_columns(evaluation, f"{field}_COMBINE", f"{field}_MANUAL")

evaluation["HAS_OFFICIAL_COMBINE"] = evaluation["PLAYER_NAME_COMBINE"].notna()
evaluation["COMBINE_SOURCE_URL"] = combine_source_url

evaluation[
    [
        "PLAYER_NAME", "POSITION", "SCHOOL_OR_TEAM", "PROSPECT_ROLE", "HAS_OFFICIAL_COMBINE",
        "HEIGHT_WO_SHOES", "WEIGHT", "WINGSPAN", "STANDING_REACH", "MAX_VERTICAL_LEAP",
        "LANE_AGILITY_TIME", "THREE_QUARTER_SPRINT", "MEASUREMENT_SOURCE", "MEASUREMENT_NOTES"
    ]
]

,PLAYER_NAME,POSITION,SCHOOL_OR_TEAM,PROSPECT_ROLE,HAS_OFFICIAL_COMBINE,HEIGHT_WO_SHOES,WEIGHT,WINGSPAN,STANDING_REACH,MAX_VERTICAL_LEAP,LANE_AGILITY_TIME,THREE_QUARTER_SPRINT,MEASUREMENT_SOURCE,MEASUREMENT_NOTES
0,AJ Dybantsa,F,BYU,primary_wing_creator,True,80.50,217.0,84.50,106.0,42.0,11.06,3.14,NaN,NaN
1,Darryn Peterson,G,Kansas,lead_guard_creator,True,76.50,198.8,81.75,103.0,37.5,11.17,3.16,nba_combine,NaN
2,Cameron Boozer,PF,Duke,frontcourt_star,True,80.25,252.8,85.50,108.0,35.0,11.06,3.31,nba_combine,NaN
3,Caleb Wilson,F,North Carolina,athletic_forward_defender,True,81.25,210.8,84.25,108.0,39.5,11.17,3.23,nba_combine,NaN
4,Koa Peat,F,Arizona,two_way_forward,True,79.00,245.0,83.25,104.0,37.5,11.00,3.16,nba_combine,NaN
5,Chris Cenac Jr.,F/C,Houston,rim_running_big,True,82.25,239.6,89.00,108.5,37.0,10.76,3.27,NaN,NaN
6,Jayden Quaintance,F/C,Kentucky,rim_protecting_big,True,81.00,253.4,89.25,109.0,NaN,NaN,NaN,nba_combine,NaN
7,Karim Lopez,F,New Zealand Breakers,international_two_way_forward,True,80.25,221.8,83.50,105.5,38.0,11.14,3.32,nba_combine,NaN
8,Aday Mara,C,Michigan,rim_protecting_big,True,87.00,259.8,90.00,117.0,28.0,11.47,3.61,nba_combine,NaN
9,Yaxel Lendeborg,F,Michigan,older_two_way_forward,True,80.75,241.4,87.25,108.5,32.0,10.82,3.35,nba_combine,NaN


## First-Pass Prospect Evaluation Scores

These scores are not final draft grades. They are transparent helper scores for comparing prospects. The physical score rewards size, length, vertical athleticism, and agility. The college production score uses whatever prospect stats are currently available in the seed table.

In [4]:
evaluation["physical_score"] = (
    z_score(evaluation["WINGSPAN"]) * 0.25
    + z_score(evaluation["STANDING_REACH"]) * 0.25
    + z_score(evaluation["WEIGHT"]) * 0.10
    + z_score(evaluation["MAX_VERTICAL_LEAP"]) * 0.15
    + z_score(evaluation["LANE_AGILITY_TIME"], lower_is_better=True) * 0.15
    + z_score(evaluation["THREE_QUARTER_SPRINT"], lower_is_better=True) * 0.10
)

evaluation["college_production_score"] = (
    z_score(evaluation["PTS"]) * 0.20
    + z_score(evaluation["REB"]) * 0.20
    + z_score(evaluation["AST"]) * 0.15
    + z_score(evaluation["STL"]) * 0.15
    + z_score(evaluation["BLK"]) * 0.15
    + z_score(evaluation["FG3_PCT"]) * 0.10
    + z_score(evaluation["FT_PCT"]) * 0.05
)
evaluation["prospect_eval_score"] = (
    evaluation["college_production_score"].fillna(0) * 0.50
    + evaluation["physical_score"].fillna(0) * 0.50
)

evaluation = evaluation.sort_values("prospect_eval_score", ascending=False).reset_index(drop=True)

evaluation[
    [
        "PLAYER_NAME", "POSITION", "DRAFT_RANGE", "PROSPECT_ROLE", "HAS_OFFICIAL_COMBINE",
        "PTS", "REB", "AST", "STL", "BLK", "FG3_PCT", "HEIGHT_WO_SHOES", "WEIGHT",
        "WINGSPAN", "STANDING_REACH", "MAX_VERTICAL_LEAP", "physical_score",
        "college_production_score", "prospect_eval_score"
    ]
]

,PLAYER_NAME,POSITION,DRAFT_RANGE,PROSPECT_ROLE,HAS_OFFICIAL_COMBINE,PTS,REB,AST,STL,BLK,FG3_PCT,HEIGHT_WO_SHOES,WEIGHT,WINGSPAN,STANDING_REACH,MAX_VERTICAL_LEAP,physical_score,college_production_score,prospect_eval_score
0,Cameron Boozer,PF,top_4,frontcourt_star,True,22.5,10.2,4.1,1.4,0.6,0.391,80.25,252.8,85.50,108.0,35.0,0.280556,0.913681,0.597118
1,Caleb Wilson,F,bulls_pick_4,athletic_forward_defender,True,19.8,9.4,2.7,1.5,1.4,0.259,81.25,210.8,84.25,108.0,39.5,0.249496,0.485755,0.367625
2,AJ Dybantsa,F,top_3,primary_wing_creator,True,25.5,6.8,3.7,1.1,0.3,0.331,80.50,217.0,84.50,106.0,42.0,0.407162,0.296829,0.351996
3,Yaxel Lendeborg,F,pick_15,older_two_way_forward,True,15.1,6.8,3.2,1.1,1.2,0.372,80.75,241.4,87.25,108.5,32.0,0.319176,0.168314,0.243745
4,Cameron Carr,G/F,pick_15,scoring_wing,True,18.9,5.8,2.4,NaN,1.3,0.370,76.50,184.4,84.75,104.0,42.5,0.454553,NaN,0.227276
5,Nate Ament,F,pick_15,stretch_forward,True,15.2,6.9,2.8,1.4,NaN,0.304,81.50,210.8,83.50,109.5,37.0,0.107517,NaN,0.053758
6,Aday Mara,C,pick_15,rim_protecting_big,True,12.1,6.8,2.4,0.4,2.6,0.300,87.00,259.8,90.00,117.0,28.0,0.360659,-0.258927,0.050866
7,Chris Cenac Jr.,F/C,bulls_pick_4,rim_running_big,True,9.5,7.9,0.7,0.8,0.5,0.333,82.25,239.6,89.00,108.5,37.0,0.727333,-0.691434,0.017949
8,Jayden Quaintance,F/C,lottery,rim_protecting_big,True,5.0,5.0,NaN,NaN,NaN,NaN,81.00,253.4,89.25,109.0,NaN,NaN,NaN,0.000000
9,Mikel Brown Jr.,G,lottery,lead_guard_creator,True,18.2,NaN,4.7,NaN,NaN,NaN,75.50,190.2,79.50,100.5,39.5,-0.274819,NaN,-0.137410


## Save Combined Evaluation Table

This saved table can be loaded by the draft/free-agency scenario notebook so draft choices can be evaluated with production and measurements together.

In [5]:
output_columns = [
    "PLAYER_NAME", "POSITION", "SCHOOL_OR_TEAM", "PROJECTED_PICK_MIN", "PROJECTED_PICK_MAX",
    "DRAFT_RANGE", "PROSPECT_ROLE", "CRAFTEDNBA_RANK", "CRAFTEDNBA_SOURCE_URL",
    "CRAFTEDNBA_SOURCE_NOTES", "STAT_SEASON", "STAT_TEAM", "GP", "MPG",
    "AGE", "PTS", "REB", "AST", "STL", "BLK", "FG_PCT", "FG3_PCT", "FT_PCT",
    "STAT_SOURCE_NAME", "STAT_SOURCE_URL", "STAT_SOURCE_NOTES",
    "HAS_OFFICIAL_COMBINE", "HEIGHT_WO_SHOES",
    "WEIGHT", "WINGSPAN", "STANDING_REACH", "MAX_VERTICAL_LEAP", "LANE_AGILITY_TIME",
    "THREE_QUARTER_SPRINT", "MEASUREMENT_SOURCE", "MEASUREMENT_NOTES",
    "COMBINE_SOURCE_URL", "physical_score", "college_production_score",
    "prospect_eval_score", "NOTES"
]

prospect_evaluation_table = evaluation[output_columns].copy()
prospect_evaluation_table.to_csv(output_path, index=False)

print(f"Saved {len(prospect_evaluation_table):,} rows to {output_path}")
prospect_evaluation_table.head(20)

Saved 18 rows to /Users/justinwahyudi/Desktop/bulls-roster-optimization/fresh_start/data/prospect_evaluation_table_2026.csv


,PLAYER_NAME,POSITION,SCHOOL_OR_TEAM,PROJECTED_PICK_MIN,PROJECTED_PICK_MAX,DRAFT_RANGE,PROSPECT_ROLE,CRAFTEDNBA_RANK,CRAFTEDNBA_SOURCE_URL,CRAFTEDNBA_SOURCE_NOTES,...,MAX_VERTICAL_LEAP,LANE_AGILITY_TIME,THREE_QUARTER_SPRINT,MEASUREMENT_SOURCE,MEASUREMENT_NOTES,COMBINE_SOURCE_URL,physical_score,college_production_score,prospect_eval_score,NOTES
0,Cameron Boozer,PF,Duke,1,4,top_4,frontcourt_star,3.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,35.0,11.06,3.31,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.280556,0.913681,0.597118,Elite production forward.
1,Caleb Wilson,F,North Carolina,3,6,bulls_pick_4,athletic_forward_defender,4.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,39.5,11.17,3.23,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.249496,0.485755,0.367625,Potential pick 4 frontcourt/wing option.
2,AJ Dybantsa,F,BYU,1,3,top_3,primary_wing_creator,2.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,42.0,11.06,3.14,NaN,NaN,https://www.nba.com/stats/draft/combine-anthro,0.407162,0.296829,0.351996,Likely unavailable at Bulls pick 4 unless he f...
3,Yaxel Lendeborg,F,Michigan,10,22,pick_15,older_two_way_forward,10.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,32.0,10.82,3.35,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.319176,0.168314,0.243745,Older forward; potentially more NBA-ready.
4,Cameron Carr,G/F,Baylor,10,22,pick_15,scoring_wing,21.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,42.5,10.46,3.17,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.454553,NaN,0.227276,Scoring wing.
5,Nate Ament,F,Tennessee,8,18,pick_15,stretch_forward,9.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,37.0,11.27,3.26,NaN,NaN,https://www.nba.com/stats/draft/combine-anthro,0.107517,NaN,0.053758,Forward with shooting/size appeal.
6,Aday Mara,C,Michigan,8,18,pick_15,rim_protecting_big,29.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,28.0,11.47,3.61,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.360659,-0.258927,0.050866,Possible pick 15 size/rim protection option.
7,Chris Cenac Jr.,F/C,Houston,4,10,bulls_pick_4,rim_running_big,18.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,37.0,10.76,3.27,NaN,NaN,https://www.nba.com/stats/draft/combine-anthro,0.727333,-0.691434,0.017949,Potential frontcourt upside target.
8,Jayden Quaintance,F/C,Kentucky,6,14,lottery,rim_protecting_big,7.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,NaN,NaN,NaN,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,NaN,NaN,0.000000,Big/defensive upside; limited 2025-26 sample d...
9,Mikel Brown Jr.,G,Louisville,5,14,lottery,lead_guard_creator,6.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,39.5,10.57,3.24,NaN,NaN,https://www.nba.com/stats/draft/combine-anthro,-0.274819,NaN,-0.137410,Lead guard prospect.


## Next Steps

- Replace placeholder measurement rows with verified sources.
- Add source links for every manually entered measurement.
- Add more complete college/international stats.
- Use `prospect_eval_score` inside the draft/free-agency scenario model.
- Test whether Bulls pick 4 should prioritize best overall prospect or strongest need fit.